# 01 — ANSS ComCat rebuild and magnitude harmonisation

Rebuilds the final modelling catalogue.

The API query starts at reported magnitude M≥2.0 so magnitude harmonisation is
performed **before** the final Mw-equivalent threshold is imposed. The final
input catalogue is then filtered to $M_w^\ast\ge2.5$.

For exact submitted-result reproduction, prefer the archived processed catalogue:
ANSS ComCat is a live catalogue and historical records can be revised.

In [1]:
from pathlib import Path
from io import StringIO
import time
import warnings

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 100)

# ---------------------------------------------------------
# Project configuration
# ---------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
AUDIT_DIR = OUTPUT_DIR / "audit"

for _d in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR, PRED_DIR, AUDIT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw_usgs_m20"
RAW_DIR.mkdir(parents=True, exist_ok=True)

GRID_MASK_FILE = DATA_DIR / "california_grid_centre_mask.csv"
REFERENCE_M25_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_M25_centre_mask.csv"

RAW_COMBINED_FILE = DATA_DIR / "usgs_bbox_2010_2025_reported_Mge2.csv"
CA_M20_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_reported_Mge2_centre_mask.csv"
HARMONISED_ALL_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_Mw_harmonised_all.csv"
FINAL_MW25_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_Mw25_centre_mask.csv"
UNRESOLVED_FILE = DATA_DIR / "magnitude_harmonisation_unresolved.csv"

# Study period and original bounding box
START_YEAR = 2010
END_YEAR = 2025
LAT_MIN, LAT_MAX = 32.0, 42.5
LON_MIN, LON_MAX = -125.0, -114.0
RAW_MIN_MAG = 2.0

# Final grid definition used by the saved centre-based mask
N_LAT = 53
N_LON = 55
LAT_EDGES = np.linspace(LAT_MIN, LAT_MAX, N_LAT + 1)
LON_EDGES = np.linspace(LON_MIN, LON_MAX, N_LON + 1)

print("Working directory:", DATA_DIR.resolve())
print("Latitude cell width:", LAT_EDGES[1] - LAT_EDGES[0])
print("Longitude cell width:", LON_EDGES[1] - LON_EDGES[0])

Working directory: E:\test\california_earthquake_forecasting_reproducible\data
Latitude cell width: 0.19811320754716633
Longitude cell width: 0.20000000000000284


## 1. Load and validate the fixed centre-based California grid mask

The spatial domain is **not rebuilt from a shapefile here**. The saved 1,080-cell mask is treated as the authoritative modelling region so that the new catalogue uses exactly the same spatial definition as the existing project.

In [2]:
if not GRID_MASK_FILE.exists():
    raise FileNotFoundError(
        f"Missing {GRID_MASK_FILE}. Put california_grid_centre_mask.csv "
        "in the same folder as this notebook."
    )

grid_mask = pd.read_csv(GRID_MASK_FILE)

required_mask_cols = {"cell_id", "lat_idx", "lon_idx", "is_california"}
missing_cols = required_mask_cols - set(grid_mask.columns)
if missing_cols:
    raise ValueError(f"Grid mask is missing columns: {sorted(missing_cols)}")

assert len(grid_mask) == 1080, f"Expected 1080 retained cells, found {len(grid_mask)}"
assert grid_mask["cell_id"].is_unique, "cell_id must be unique in the grid mask"
assert grid_mask["is_california"].fillna(False).all(), "Mask contains non-California cells"

valid_cell_ids = set(grid_mask["cell_id"].astype(int))

print("Retained centre-based California cells:", len(grid_mask))
print("cell_id range:", grid_mask["cell_id"].min(), "to", grid_mask["cell_id"].max())
display(grid_mask.head())

Retained centre-based California cells: 1080
cell_id range: 203 to 2764


,cell_id,lat_idx,lon_idx,south,north,west,east,centre_lat,centre_lon,is_california
0,203,3,38,32.59434,32.792453,-117.4,-117.2,32.693396,-117.3,True
1,204,3,39,32.59434,32.792453,-117.2,-117.0,32.693396,-117.1,True
2,205,3,40,32.59434,32.792453,-117.0,-116.8,32.693396,-116.9,True
3,206,3,41,32.59434,32.792453,-116.8,-116.6,32.693396,-116.7,True
4,207,3,42,32.59434,32.792453,-116.6,-116.4,32.693396,-116.5,True


## 2. Download USGS ComCat data (`minmagnitude = 2.0`) with caching

USGS FDSN Event Web Service limits a single query to 20,000 results, so each calendar year is downloaded with pagination. Each yearly file is cached locally. If a cache exists, rerunning this notebook loads it rather than querying the API again.

API documentation: USGS FDSN Event Web Service, `query`, `limit`, `offset`, bounding-box and magnitude parameters.

In [3]:
USGS_QUERY_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
PAGE_LIMIT = 20000


def download_usgs_year(year, force=False, max_retries=4):
    """Download one year of the bounding-box catalogue, using pagination and a local cache."""
    year_file = RAW_DIR / f"usgs_bbox_Mge2_{year}.csv"

    if year_file.exists() and not force:
        out = pd.read_csv(year_file)
        print(f"{year}: loaded cache ({len(out):,} events)")
        return out

    session = requests.Session()
    chunks = []
    offset = 1

    while True:
        params = {
            "format": "csv",
            "starttime": f"{year}-01-01T00:00:00",
            "endtime": f"{year}-12-31T23:59:59.999",
            "minlatitude": LAT_MIN,
            "maxlatitude": LAT_MAX,
            "minlongitude": LON_MIN,
            "maxlongitude": LON_MAX,
            "minmagnitude": RAW_MIN_MAG,
            "eventtype": "earthquake",
            "orderby": "time-asc",
            "limit": PAGE_LIMIT,
            "offset": offset,
        }

        response = None
        for attempt in range(1, max_retries + 1):
            try:
                response = session.get(USGS_QUERY_URL, params=params, timeout=120)
                response.raise_for_status()
                break
            except requests.RequestException as exc:
                if attempt == max_retries:
                    raise
                wait = 2 ** (attempt - 1)
                print(f"  request failed ({exc}); retrying in {wait}s")
                time.sleep(wait)

        if response.status_code == 204 or not response.text.strip():
            break

        chunk = pd.read_csv(StringIO(response.text))
        if chunk.empty:
            break

        chunks.append(chunk)
        print(f"{year}: offset {offset:,} -> {len(chunk):,} events")

        if len(chunk) < PAGE_LIMIT:
            break

        # offset is 1-based
        offset += len(chunk)

    if not chunks:
        out = pd.DataFrame()
    else:
        out = pd.concat(chunks, ignore_index=True)

    out.to_csv(year_file, index=False)
    print(f"{year}: saved cache -> {year_file} ({len(out):,} events)")
    return out

In [4]:
# Set FORCE_REDOWNLOAD=True only if you deliberately want to refresh all cached API data.
FORCE_REDOWNLOAD = False

if RAW_COMBINED_FILE.exists() and not FORCE_REDOWNLOAD:
    df_raw = pd.read_csv(RAW_COMBINED_FILE)
    print(f"Loaded combined raw cache: {len(df_raw):,} events")
else:
    yearly = [
        download_usgs_year(year, force=FORCE_REDOWNLOAD)
        for year in range(START_YEAR, END_YEAR + 1)
    ]
    df_raw = pd.concat(yearly, ignore_index=True)
    df_raw.to_csv(RAW_COMBINED_FILE, index=False)
    print(f"Saved combined raw catalogue: {RAW_COMBINED_FILE}")

print("Raw bounding-box events:", f"{len(df_raw):,}")
print("Raw reported magnitude range:", df_raw["mag"].min(), "to", df_raw["mag"].max())
display(df_raw.head())

2010: offset 1 -> 12,034 events
2010: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2010.csv (12,034 events)
2011: offset 1 -> 3,920 events
2011: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2011.csv (3,920 events)
2012: offset 1 -> 3,318 events
2012: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2012.csv (3,318 events)
2013: offset 1 -> 3,095 events
2013: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2013.csv (3,095 events)
2014: offset 1 -> 3,800 events
2014: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2014.csv (3,800 events)
2015: offset 1 -> 3,381 events
2015: saved cache -> e:\test\california_earthquake_forecasting_reproducible\data\raw_usgs_m20\usgs_bbox_Mge2_2015.csv (3,381 events)
2016: offset 1

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2010-01-01T02:27:45.960Z,32.470667,-115.214000,5.995,2.18,ml,18.0,204.0,0.3301,0.39,ci,ci14566868,2016-03-10T10:12:05.696Z,"13km N of Delta, B.C., MX",earthquake,1.42,31.61,0.162,27.0,reviewed,ci,ci
1,2010-01-01T02:33:42.820Z,32.453833,-115.202833,5.995,3.24,ml,28.0,207.0,0.3387,0.48,ci,ci14566876,2022-08-05T23:20:01.348Z,"11km N of Delta, B.C., MX",earthquake,1.53,31.61,0.161,253.0,reviewed,ci,ci
2,2010-01-01T02:55:04.370Z,35.985167,-117.298333,0.658,2.79,ml,38.0,59.0,0.0948,0.23,ci,ci14566884,2016-03-10T01:14:18.198Z,"26km NNE of Trona, CA",earthquake,0.44,0.61,0.158,237.0,reviewed,ci,ci
3,2010-01-01T03:25:30.180Z,36.032500,-117.782833,0.713,2.86,ml,37.0,40.0,0.0135,0.22,ci,ci14566908,2016-03-10T17:46:14.902Z,"15km E of Coso Junction, CA",earthquake,0.30,0.49,0.161,93.0,reviewed,ci,ci
4,2010-01-01T06:52:51.400Z,32.465333,-115.215000,5.995,2.56,ml,21.0,208.0,0.3350,0.33,ci,ci14567060,2016-03-10T04:14:51.424Z,"12km N of Delta, B.C., MX",earthquake,1.09,31.61,0.158,120.0,reviewed,ci,ci


## 3. Basic cleaning and audit

Only fields essential to the present modelling catalogue are required at this stage: event ID, time, location and reported magnitude. Magnitude type is **not** required for this cleaning step because unsupported or missing magnitude types are handled explicitly during harmonisation rather than silently removed here.

In [5]:
df_clean = df_raw.copy()

# Normalise labels before any magnitude-type logic.
df_clean["magType"] = (
    df_clean["magType"].astype("string").str.strip().str.lower()
)
df_clean["magSource"] = (
    df_clean["magSource"].astype("string").str.strip().str.lower()
)
df_clean["net"] = (
    df_clean["net"].astype("string").str.strip().str.lower()
)

# Mixed timestamps occur in ComCat CSVs (some include fractional seconds, some do not).
df_clean["time"] = pd.to_datetime(
    df_clean["time"], format="mixed", utc=True, errors="coerce"
)
if "updated" in df_clean.columns:
    df_clean["updated"] = pd.to_datetime(
        df_clean["updated"], format="mixed", utc=True, errors="coerce"
    )

df_clean["year"] = df_clean["time"].dt.year

critical = ["id", "time", "latitude", "longitude", "mag"]
missing_critical = df_clean[critical].isna().any(axis=1)
invalid_coords = ~(
    df_clean["latitude"].between(-90, 90)
    & df_clean["longitude"].between(-180, 180)
)
wrong_year = ~df_clean["year"].between(START_YEAR, END_YEAR)
wrong_type = (
    df_clean["type"].ne("earthquake")
    if "type" in df_clean.columns
    else pd.Series(False, index=df_clean.index)
)

print("Raw rows:", len(df_clean))
print("Rows with missing critical fields:", int(missing_critical.sum()))
print("Rows with invalid coordinates:", int(invalid_coords.sum()))
print("Rows outside study years:", int(wrong_year.sum()))
print("Rows not labelled earthquake:", int(wrong_type.sum()))
print("Duplicate event IDs:", int(df_clean["id"].duplicated().sum()))

# Remove only observations that cannot be used reliably in the spatial/time catalogue.
df_clean = df_clean.loc[
    ~(missing_critical | invalid_coords | wrong_year | wrong_type)
].copy()

# If the service ever returns duplicate IDs, retain the latest record.
if "updated" in df_clean.columns:
    df_clean = df_clean.sort_values(["id", "updated"])
df_clean = df_clean.drop_duplicates(subset="id", keep="last").reset_index(drop=True)

print("Rows after cleaning:", f"{len(df_clean):,}")

Raw rows: 72167
Rows with missing critical fields: 0
Rows with invalid coordinates: 0
Rows outside study years: 0
Rows not labelled earthquake: 0
Duplicate event IDs: 0
Rows after cleaning: 72,167


## 4. Assign events to the established 53 × 55 grid and apply the 1,080-cell mask

The original bounding box is divided into 53 latitude intervals and 55 longitude intervals using the same `np.linspace` edges as the final project grid. Event indices are assigned using `np.searchsorted(..., side="right") - 1`, and

\[
\text{cell\_id}=i_{\text{lat}}\,n_{\text{lon}}+i_{\text{lon}}.
\]

An event is retained only if its `cell_id` occurs in the saved centre-based California mask.

In [6]:
lat_idx = np.searchsorted(
    LAT_EDGES, df_clean["latitude"].to_numpy(), side="right"
) - 1
lon_idx = np.searchsorted(
    LON_EDGES, df_clean["longitude"].to_numpy(), side="right"
) - 1

valid_grid_index = (
    (lat_idx >= 0) & (lat_idx < N_LAT)
    & (lon_idx >= 0) & (lon_idx < N_LON)
)

# Only rows with valid indices can receive a cell ID.
df_grid = df_clean.loc[valid_grid_index].copy()
df_grid["lat_idx"] = lat_idx[valid_grid_index]
df_grid["lon_idx"] = lon_idx[valid_grid_index]
df_grid["cell_id"] = (
    df_grid["lat_idx"] * N_LON + df_grid["lon_idx"]
).astype(int)

df_grid["cell_in_mask"] = df_grid["cell_id"].isin(valid_cell_ids)
df_ca = df_grid.loc[df_grid["cell_in_mask"]].copy().reset_index(drop=True)

print("Clean bounding-box events:", f"{len(df_clean):,}")
print("Events assigned inside the 53x55 grid:", f"{len(df_grid):,}")
print("Events retained by 1,080-cell centre mask:", f"{len(df_ca):,}")
print("Occupied retained cells:", df_ca["cell_id"].nunique())

assert df_ca["cell_id"].isin(valid_cell_ids).all()
assert df_ca["cell_in_mask"].all()

df_ca.to_csv(CA_M20_FILE, index=False)
print("Saved pre-harmonisation California catalogue:", CA_M20_FILE)

Clean bounding-box events: 72,167
Events assigned inside the 53x55 grid: 72,167
Events retained by 1,080-cell centre mask: 48,787
Occupied retained cells: 830
Saved pre-harmonisation California catalogue: e:\test\california_earthquake_forecasting_reproducible\data\earthquake_california_grid_2010_2025_reported_Mge2_centre_mask.csv


## 5. Magnitude-type audit before harmonisation

The conversion rules are deliberately limited to the dominant magnitude/network combinations for which a chosen literature-supported relation is available. Other combinations remain labelled `unresolved` and are reported explicitly.

In [8]:
magtype_summary = (
    df_ca["magType"]
    .value_counts(dropna=False)
    .rename_axis("magType")
    .reset_index(name="count")
)
magtype_summary["percentage"] = (
    magtype_summary["count"] / len(df_ca) * 100
).round(2)

display(magtype_summary)

source_summary = (
    df_ca.groupby(["magType", "net", "magSource"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(source_summary.head(30))

,magType,count,percentage
0,ml,26518,54.35
1,md,20567,42.16
2,mw,1262,2.59
3,mlr,252,0.52
4,mh,184,0.38
5,mb,2,0.0
6,<NA>,1,0.0
7,mwr,1,0.0


,magType,net,magSource,count
5,ml,ci,ci,24877
1,md,nc,nc,20566
6,ml,nc,nc,1376
17,mw,nc,nc,755
15,mw,ci,ci,506
14,mlr,ci,ci,252
7,ml,nn,nn,223
3,mh,ci,ci,97
4,mh,nc,nc,87
12,ml,us,us,27


## 6. Harmonise supported magnitudes to an Mw-equivalent scale

### Working conversion policy

- Native moment-magnitude family (`mw`, `mwr`, `mww`, `mwb`, `mwc`): retain the reported value.
- Southern California (`ci`) `ml`: use the Southern California/Ridgecrest local-magnitude–moment relation of **Baltay & Abercrombie (2025)**, then convert seismic moment to `Mw`.
- Southern California (`ci`) `mlr`: invert the SCSN revised-local-magnitude relation to recover `ml`, then apply the same Southern California conversion.
- Northern California (`nc`) `ml`: use the central-California seismic-moment relations reported by **Bakun (1984)**.
- Northern California (`nc`) `md`: use Bakun's coda-duration relation only within its published range `1 <= Md <= 3.5`.
- Other magnitude/source combinations: leave unresolved rather than apply an unsupported conversion.

The moment-magnitude definition follows **Hanks & Kanamori (1979)**. These empirical conversions introduce uncertainty; the original magnitude and conversion method are therefore retained in the data.

In [9]:
# ---------------------------------------------------------
# Conversion functions
# ---------------------------------------------------------

def moment_to_mw(log10_m0_nm):
    """Hanks & Kanamori (1979), with M0 in N m."""
    return (2.0 / 3.0) * (log10_m0_nm - 9.05)


def ci_ml_to_mw(ml):
    """Southern California ML -> Mw-equivalent; Baltay & Abercrombie (2025)."""
    ml = np.asarray(ml, dtype=float)
    log10_m0_nm = (
        1.5 * ml
        + 0.5 * np.logaddexp(0.0, 4.6 - ml)
        + 8.35
    )
    return moment_to_mw(log10_m0_nm)


def ci_mlr_to_mw(mlr):
    """Invert SCSN MLr = 0.853 ML + 0.40125, then use CI ML conversion."""
    mlr = np.asarray(mlr, dtype=float)
    ml = (mlr - 0.40125) / 0.853
    return ci_ml_to_mw(ml)


def nc_ml_to_mw(ml):
    """Central/Northern California ML -> Mw-equivalent using Bakun (1984)."""
    ml = np.asarray(ml, dtype=float)

    # Bakun reports overlapping magnitude-dependent relations.
    # We use the lower-magnitude relation through ML=3.5 and the
    # higher-magnitude relation above 3.5.
    log10_m0_dyn_cm = np.where(
        ml <= 3.5,
        1.2 * ml + 17.0,
        1.5 * ml + 16.0,
    )

    # 1 N m = 10^7 dyne cm
    log10_m0_nm = log10_m0_dyn_cm - 7.0
    return moment_to_mw(log10_m0_nm)


def nc_md_to_mw(md):
    """Northern California Md -> Mw-equivalent; Bakun (1984), valid for 1<=Md<=3.5."""
    md = np.asarray(md, dtype=float)
    log10_m0_dyn_cm = 1.2 * md + 17.0
    log10_m0_nm = log10_m0_dyn_cm - 7.0
    return moment_to_mw(log10_m0_nm)

In [10]:
df_h = df_ca.copy()

df_h["mag_original"] = df_h["mag"]
df_h["magType_original"] = df_h["magType"]
df_h["mag_Mw"] = np.nan
df_h["mag_conversion"] = "unresolved"

# Native moment-magnitude family
native_mw_types = {"mw", "mwr", "mww", "mwb", "mwc"}
native_mw = df_h["magType"].isin(native_mw_types)
df_h.loc[native_mw, "mag_Mw"] = df_h.loc[native_mw, "mag"]
df_h.loc[native_mw, "mag_conversion"] = "native_Mw_family"

# Southern California ML
ci_ml = (df_h["magType"] == "ml") & (df_h["magSource"] == "ci")
df_h.loc[ci_ml, "mag_Mw"] = ci_ml_to_mw(df_h.loc[ci_ml, "mag"])
df_h.loc[ci_ml, "mag_conversion"] = "BaltayAbercrombie2025_CI_ML"

# Southern California revised ML
ci_mlr = (df_h["magType"] == "mlr") & (df_h["magSource"] == "ci")
df_h.loc[ci_mlr, "mag_Mw"] = ci_mlr_to_mw(df_h.loc[ci_mlr, "mag"])
df_h.loc[ci_mlr, "mag_conversion"] = "SCEDC_MLr_then_Baltay2025"

# Northern California ML
nc_ml = (df_h["magType"] == "ml") & (df_h["magSource"] == "nc")
df_h.loc[nc_ml, "mag_Mw"] = nc_ml_to_mw(df_h.loc[nc_ml, "mag"])
df_h.loc[nc_ml, "mag_conversion"] = "Bakun1984_NC_ML"

# Northern California MD, within published range only
nc_md_valid = (
    (df_h["magType"] == "md")
    & (df_h["magSource"] == "nc")
    & df_h["mag"].between(1.0, 3.5)
)
df_h.loc[nc_md_valid, "mag_Mw"] = nc_md_to_mw(df_h.loc[nc_md_valid, "mag"])
df_h.loc[nc_md_valid, "mag_conversion"] = "Bakun1984_NC_MD"

nc_md_outside = (
    (df_h["magType"] == "md")
    & (df_h["magSource"] == "nc")
    & ~df_h["mag"].between(1.0, 3.5)
)
df_h.loc[nc_md_outside, "mag_conversion"] = "MD_outside_Bakun_range"

print("Magnitude harmonisation applied.")

Magnitude harmonisation applied.


## 7. Audit conversion coverage before applying any final threshold

This checkpoint is important: unresolved observations are not silently converted or silently ignored. Their magnitude type and reporting source are shown before the final catalogue is constructed.

In [11]:
conversion_summary = (
    df_h["mag_conversion"]
    .value_counts(dropna=False)
    .rename_axis("method")
    .reset_index(name="count")
)
conversion_summary["percentage"] = (
    conversion_summary["count"] / len(df_h) * 100
).round(2)

display(conversion_summary)

unresolved = df_h.loc[df_h["mag_Mw"].isna()].copy()
unresolved_summary = (
    unresolved.groupby(["magType", "net", "magSource"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(unresolved_summary)

unresolved_pct = 100 * len(unresolved) / len(df_h)
print(f"Unresolved: {len(unresolved):,} / {len(df_h):,} ({unresolved_pct:.2f}%)")
print("NC Md outside Bakun range:", int(nc_md_outside.sum()))

if unresolved_pct > 1.0:
    warnings.warn(
        "More than 1% of the centre-mask catalogue is unresolved. "
        "Review unresolved_summary before treating the final Mw>=2.5 catalogue as fixed."
    )

,method,count,percentage
0,BaltayAbercrombie2025_CI_ML,24882,51.00
1,Bakun1984_NC_MD,20562,42.15
2,Bakun1984_NC_ML,1377,2.82
3,native_Mw_family,1263,2.59
4,unresolved,447,0.92
5,SCEDC_MLr_then_Baltay2025,252,0.52
6,MD_outside_Bakun_range,4,0.01


,magType,net,magSource,count
5,ml,nn,nn,223
3,mh,ci,ci,97
4,mh,nc,nc,87
8,ml,us,us,27
1,md,nc,nc,4
7,ml,us,ren,4
9,ml,uw,uw,3
0,mb,nn,nn,2
6,ml,us,pas,2
2,md,uw,uw,1


Unresolved: 451 / 48,787 (0.92%)
NC Md outside Bakun range: 4


## 8. Quantify the effect of harmonisation on the 2.5 and 3.0 thresholds

The first comparison answers whether the wider M ≥ 2.0 download was necessary: it counts events with reported magnitude below 2.5 that move to `Mw >= 2.5` after harmonisation. The second comparison records how many observations change classification around 3.0; this will be used later when deciding and justifying the forecasting target threshold.

In [12]:
resolved = df_h["mag_Mw"].notna()

added_across_25 = (
    resolved
    & (df_h["mag_original"] < 2.5)
    & (df_h["mag_Mw"] >= 2.5)
)
lost_across_25 = (
    resolved
    & (df_h["mag_original"] >= 2.5)
    & (df_h["mag_Mw"] < 2.5)
)

print("Reported M<2.5 but harmonised Mw>=2.5:", int(added_across_25.sum()))
print("Reported M>=2.5 but harmonised Mw<2.5:", int(lost_across_25.sum()))

threshold_25 = pd.crosstab(
    df_h.loc[resolved, "mag_original"] >= 2.5,
    df_h.loc[resolved, "mag_Mw"] >= 2.5,
    rownames=["Original reported M >= 2.5"],
    colnames=["Harmonised Mw >= 2.5"],
)
display(threshold_25)

threshold_30 = pd.crosstab(
    df_h.loc[resolved, "mag_original"] >= 3.0,
    df_h.loc[resolved, "mag_Mw"] >= 3.0,
    rownames=["Original reported M >= 3.0"],
    colnames=["Harmonised Mw >= 3.0"],
)
display(threshold_30)

m3_changed = (
    (df_h.loc[resolved, "mag_original"] >= 3.0)
    != (df_h.loc[resolved, "mag_Mw"] >= 3.0)
)
print("Events changing M=3.0 classification:", int(m3_changed.sum()))
print("Percentage of resolved events:", f"{100*m3_changed.mean():.2f}%")

Reported M<2.5 but harmonised Mw>=2.5: 13987
Reported M>=2.5 but harmonised Mw<2.5: 0


Harmonised Mw >= 2.5,False,True
Original reported M >= 2.5,,
False,17996,13987
True,0,16353


Harmonised Mw >= 3.0,False,True
Original reported M >= 3.0,,
False,41750,1592
True,0,4994


Events changing M=3.0 classification: 1592
Percentage of resolved events: 3.29%


## 9. Construct and save the final Mw ≥ 2.5 modelling catalogue

Events without a supported magnitude conversion are saved separately. The working final catalogue contains only observations with a resolved `Mw`-equivalent magnitude satisfying `Mw >= 2.5`.

This exclusion rule must be reported together with the unresolved count and percentage. If the unresolved proportion is materially larger than expected, revisit the conversion policy before modelling.

In [16]:
# Save every centre-mask event with the conversion fields for full traceability.
df_h.to_csv(HARMONISED_ALL_FILE, index=False)
unresolved.to_csv(UNRESOLVED_FILE, index=False)

# Rebuilt catalogue from the current source data:
# supported harmonisation + Mw >= 2.5.
df_final = df_h.loc[
    df_h["mag_Mw"].notna() & (df_h["mag_Mw"] >= 2.5)
].copy().reset_index(drop=True)

# Keep a clear working magnitude variable while preserving all original columns.
df_final["mag_model"] = df_final["mag_Mw"]



REBUILT_MW25_FILE = (
    OUTPUT_DIR
    / "audit"
    / "earthquake_california_grid_2010_2025_Mw25_rebuilt.csv"
)

REBUILT_MW25_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_final.to_csv(
    REBUILT_MW25_FILE,
    index=False
)


print(
    "Saved all harmonised/flagged centre-mask events:",
    HARMONISED_ALL_FILE
)

print(
    "Saved unresolved events:",
    UNRESOLVED_FILE
)

print(
    "Saved rebuilt Mw>=2.5 catalogue for audit:",
    REBUILT_MW25_FILE
)

print()

print(
    "Rebuilt events:",
    f"{len(df_final):,}"
)

print(
    "Occupied cells:",
    df_final["cell_id"].nunique()
)

print(
    "Full modelling domain remains:",
    len(grid_mask),
    "cells"
)

print(
    "Magnitude range (Mw-equivalent):",
    df_final["mag_Mw"].min(),
    "to",
    df_final["mag_Mw"].max()
)

Saved all harmonised/flagged centre-mask events: e:\test\california_earthquake_forecasting_reproducible\data\earthquake_california_grid_2010_2025_Mw_harmonised_all.csv
Saved unresolved events: e:\test\california_earthquake_forecasting_reproducible\data\magnitude_harmonisation_unresolved.csv
Saved rebuilt Mw>=2.5 catalogue for audit: e:\test\california_earthquake_forecasting_reproducible\outputs\audit\earthquake_california_grid_2010_2025_Mw25_rebuilt.csv

Rebuilt events: 30,340
Occupied cells: 746
Full modelling domain remains: 1080 cells
Magnitude range (Mw-equivalent): 2.5053333333333327 to 7.1


## 10. Save audit tables for the report

These small CSVs make it easy to quote exact numbers later without rerunning the full notebook.

In [14]:
conversion_summary.to_csv(DATA_DIR / "audit_magnitude_conversion_summary.csv", index=False)
unresolved_summary.to_csv(DATA_DIR / "audit_magnitude_unresolved_summary.csv", index=False)
threshold_25.to_csv(DATA_DIR / "audit_threshold_25_crossing.csv")
threshold_30.to_csv(DATA_DIR / "audit_threshold_30_crossing.csv")
magtype_summary.to_csv(DATA_DIR / "audit_magtype_summary_Mge2_centre_mask.csv", index=False)
source_summary.to_csv(DATA_DIR / "audit_magtype_network_source_Mge2_centre_mask.csv", index=False)

print("Audit tables saved.")

Audit tables saved.


## 11. Set the rebuilt catalogue as the working `df`

From this point onward, subsequent EDA, daily grid counts, predictor construction and target construction should use the harmonised `Mw >= 2.5` catalogue below. The target magnitude threshold should **not** be hard-coded until the planned threshold sensitivity check is completed.

In [15]:
df = df_final.copy()

print("Working df:", df.shape)
print("Dates:", df["time"].min(), "to", df["time"].max())
print("Grid cells with at least one event:", df["cell_id"].nunique())
display(df.head())

Working df: (30340, 32)
Dates: 2010-01-01 02:55:04.370000+00:00 to 2025-12-31 15:28:39+00:00
Grid cells with at least one event: 746


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,year,lat_idx,lon_idx,cell_id,cell_in_mask,mag_original,magType_original,mag_Mw,mag_conversion,mag_model
0,2010-03-01 10:40:00.040000+00:00,36.065333,-117.881667,0.590,2.75,ml,17.0,184.0,0.10000,0.27,ci,ci10144402,2016-03-10 08:47:15.609000+00:00,"6km ENE of Coso Junction, CA",earthquake,0.77,0.65,0.147,171.0,reviewed,ci,ci,2010,20,35,1135,True,2.75,ml,2.948678,BaltayAbercrombie2025_CI_ML,2.948678
1,2010-04-23 02:26:33.090000+00:00,32.734167,-115.880000,2.228,2.55,ml,19.0,43.0,0.02560,0.10,ci,ci10146342,2016-03-10 21:48:19.410000+00:00,"11km E of Ocotillo, CA",earthquake,0.28,0.49,0.130,75.0,reviewed,ci,ci,2010,3,45,210,True,2.55,ml,2.807032,BaltayAbercrombie2025_CI_ML,2.807032
2,2010-04-22 19:16:55.300000+00:00,32.660833,-115.786167,9.145,2.18,ml,20.0,134.0,0.01930,0.15,ci,ci10146566,2016-03-10 23:20:53.009000+00:00,"21km WNW of Progreso, B.C., MX",earthquake,0.55,0.47,0.211,26.0,reviewed,ci,ci,2010,3,46,211,True,2.18,ml,2.548396,BaltayAbercrombie2025_CI_ML,2.548396
3,2010-04-04 23:36:20.700000+00:00,32.653333,-115.751667,4.553,3.14,ml,13.0,168.0,0.02170,0.18,ci,ci10147522,2016-03-10 23:02:24.811000+00:00,"18km WNW of Progreso, B.C., MX",earthquake,0.90,0.58,0.134,35.0,reviewed,ci,ci,2010,3,46,211,True,3.14,ml,3.229610,BaltayAbercrombie2025_CI_ML,3.229610
4,2010-04-04 23:47:20.590000+00:00,32.624833,-115.743667,1.973,3.00,ml,7.0,225.0,0.02845,0.16,ci,ci10147762,2016-03-10 07:22:41.555000+00:00,"16km WNW of Progreso, B.C., MX",earthquake,1.37,0.77,0.121,25.0,reviewed,ci,ci,2010,3,46,211,True,3.00,ml,3.127967,BaltayAbercrombie2025_CI_ML,3.127967
